In [ ]:
!pip -q install requests beautifulsoup4 lxml tqdm

import os,re,json,time,random,sqlite3,threading,csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
from concurrent.futures import ThreadPoolExecutor,as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

MAIN_CATEGORY="اقتصاد"
TARGET=100000
WORKERS=20
DB_BATCH=2000
CHECKPOINT=2000
TIMEOUT=15
MIN_DELAY=.03
MAX_DELAY=.10
MAX_ARCHIVE_PAGES=15000

from google.colab import drive
drive.mount("/content/drive")

BASE="/content/drive/MyDrive/AL_AIN_100K"
os.makedirs(BASE,exist_ok=True)

DB_PATH=os.path.join(BASE,f"{MAIN_CATEGORY}.db")
CSV_PATH=os.path.join(BASE,f"{MAIN_CATEGORY}_100K.csv")

TAX={
"الأخبار":{
"سوشيال ميديا":"https://al-ain.com/tag/trend/",
"الخليج العربي":"https://al-ain.com/tag/28716/",
"الوطن العربي":"https://al-ain.com/tag/20501/",
"شؤون دولية":"https://al-ain.com/tag/14651/"},
"سياسة":{
"التحليلات":"https://al-ain.com/section/analytics/",
"المغرب العربي":"https://al-ain.com/tag/28186/",
"شؤون أفريقية":"https://al-ain.com/tag/african-union/",
"الإسلام السياسي":"https://al-ain.com/tag/22966/",
"اليمن":"https://al-ain.com/tag/21476/",
"العراق":"https://al-ain.com/tag/24781/",
"ليبيا":"https://al-ain.com/tag/libyan-affairs/"},
"اقتصاد":{
"عملات رقمية":"https://al-ain.com/tag/cryptocurrency-prices/",
"تكنولوجيا":"https://al-ain.com/section/technology/",
"سياحة وسفر":"https://al-ain.com/section/travel/",
"سيارات":"https://al-ain.com/section/cars/",
"الذكاء الاصطناعي":"https://al-ain.com/section/artificial-intelligence/",
"الطاقة المتجددة":"https://al-ain.com/tag/32536/"},
"منوعات":{
"صحة":"https://al-ain.com/section/health/",
"ثقافة":"https://al-ain.com/section/culture/",
"فن":"https://al-ain.com/section/arts/",
"شباب":"https://al-ain.com/section/youth/",
"علوم وفضاء":"https://al-ain.com/section/science/",
"حوادث":"https://al-ain.com/section/crime/",
"بيئة":"https://al-ain.com/section/environment/"},
"رياضة":{
"رياضة":"https://al-ain.com/section/sports/"}}

if MAIN_CATEGORY not in TAX:
    raise ValueError("MAIN_CATEGORY غير صحيح")

SOURCES=TAX[MAIN_CATEGORY]

db=sqlite3.connect(DB_PATH,timeout=60,check_same_thread=False)
db.execute("PRAGMA journal_mode=WAL")
db.execute("PRAGMA synchronous=NORMAL")
db.execute("PRAGMA busy_timeout=60000")

db.execute("""
CREATE TABLE IF NOT EXISTS articles(
url TEXT PRIMARY KEY,
title TEXT,
main_category TEXT,
subcategory TEXT,
tags TEXT,
date TEXT,
scraped_at TEXT)
""")

db.execute("""
CREATE TABLE IF NOT EXISTS archive_state(
source TEXT PRIMARY KEY,
next_page INTEGER,
finished INTEGER DEFAULT 0)
""")

db.commit()
LOCK=threading.Lock()
LOCAL=threading.local()

def get_session():
    if not hasattr(LOCAL,"session"):
        s=requests.Session()
        retry=Retry(
            total=5,
            connect=5,
            read=5,
            backoff_factor=.8,
            status_forcelist=[429,500,502,503,504],
            allowed_methods=["GET"],
            respect_retry_after_header=True)
        adapter=HTTPAdapter(
            max_retries=retry,
            pool_connections=WORKERS,
            pool_maxsize=WORKERS)
        s.mount("https://",adapter)
        s.mount("http://",adapter)
        s.headers.update({
            "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
            "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language":"ar,en-US;q=0.9,en;q=0.8"})
        LOCAL.session=s
    return LOCAL.session

def fetch(url):
    time.sleep(random.uniform(MIN_DELAY,MAX_DELAY))
    try:
        r=get_session().get(
            url,
            timeout=TIMEOUT,
            allow_redirects=True)
        return r.status_code,r.text,r.url
    except Exception:
        return 0,"",url

def normalize(url):
    try:
        p=urlparse(url.split("#")[0].strip())
        if p.netloc.lower() not in ("al-ain.com","www.al-ain.com"):
            return None
        if "/article/" not in p.path.lower():
            return None
        return "https://al-ain.com"+p.path.rstrip("/")
    except:
        return None

def archive_url(source,page):
    source=source.rstrip("/")
    return source+"/" if page==1 else f"{source}/page-{page}.html"

def clean(x):
    return re.sub(r"\s+"," ",str(x or "")).strip()

def extract_links(html,current_url):
    soup=BeautifulSoup(html,"lxml")
    out=set()
    for a in soup.find_all("a",href=True):
        u=normalize(urljoin(current_url,a["href"]))
        if u:
            out.add(u)
    return out

def parse_article(url,subcategories):
    status,html,final_url=fetch(url)
    if status!=200 or not html:
        return None

    soup=BeautifulSoup(html,"lxml")

    title=""
    x=soup.find("meta",property="og:title")
    if x:
        title=x.get("content","")
    if not title:
        x=soup.find("h1")
        if x:
            title=x.get_text(" ",strip=True)

    date=""
    for tag,attrs in [
        ("meta",{"property":"article:published_time"}),
        ("meta",{"name":"date"}),
        ("meta",{"name":"publication_date"}),
        ("meta",{"property":"article:modified_time"})]:
        x=soup.find(tag,attrs)
        if x and x.get("content"):
            date=x.get("content")
            break

    if not date:
        for x in soup.find_all("script",type="application/ld+json"):
            try:
                data=json.loads(x.string or x.get_text())
                items=data if isinstance(data,list) else [data]
                for item in items:
                    if isinstance(item,dict):
                        date=item.get("datePublished") or item.get("dateCreated") or ""
                        if date:
                            break
                if date:
                    break
            except:
                pass

    tags=[]

    for x in soup.find_all("meta",property="article:tag"):
        v=clean(x.get("content"))
        if v:
            tags.append(v)

    if not tags:
        for x in soup.find_all("script",type="application/ld+json"):
            try:
                data=json.loads(x.string or x.get_text())
                items=data if isinstance(data,list) else [data]
                for item in items:
                    if isinstance(item,dict):
                        k=item.get("keywords")
                        if isinstance(k,list):
                            tags.extend(clean(v) for v in k)
                        elif isinstance(k,str):
                            tags.extend(clean(v) for v in k.split(","))
            except:
                pass

    tags=list(dict.fromkeys(x for x in tags if x))

    return {
        "url":normalize(final_url) or url,
        "title":clean(title),
        "main_category":MAIN_CATEGORY,
        "subcategory":" | ".join(dict.fromkeys(subcategories)),
        "tags":" | ".join(tags),
        "date":clean(date),
        "scraped_at":time.strftime("%Y-%m-%d %H:%M:%S")
    }

def count_articles():
    with LOCK:
        return db.execute(
            "SELECT COUNT(*) FROM articles").fetchone()[0]

def existing_urls(urls):
    urls=list(urls)
    result=set()
    with LOCK:
        for i in range(0,len(urls),500):
            batch=urls[i:i+500]
            q=",".join("?"*len(batch))
            rows=db.execute(
                f"SELECT url FROM articles WHERE url IN ({q})",
                batch).fetchall()
            result.update(x[0] for x in rows)
    return result

def get_state(source):
    with LOCK:
        row=db.execute(
            "SELECT next_page,finished FROM archive_state WHERE source=?",
            (source,)).fetchone()
    return row if row else (1,0)

def set_state(source,page,finished=0):
    with LOCK:
        db.execute("""
        INSERT INTO archive_state(source,next_page,finished)
        VALUES(?,?,?)
        ON CONFLICT(source) DO UPDATE SET
        next_page=excluded.next_page,
        finished=excluded.finished
        """,(source,page,finished))
        db.commit()

def save_batch(rows):
    if not rows:
        return
    with LOCK:
        db.executemany("""
        INSERT OR IGNORE INTO articles
        (url,title,main_category,subcategory,tags,date,scraped_at)
        VALUES
        (:url,:title,:main_category,:subcategory,:tags,:date,:scraped_at)
        """,rows)
        db.commit()

def export_csv():
    with LOCK:
        rows=db.execute("""
        SELECT url,title,main_category,subcategory,tags,date,scraped_at
        FROM articles
        """)
        with open(CSV_PATH,"w",encoding="utf-8-sig",newline="") as f:
            w=csv.writer(f)
            w.writerow([
                "url","title","main_category",
                "subcategory","tags","date","scraped_at"])
            w.writerows(rows)

start=time.time()
total=count_articles()

print("="*90)
print("AL-AIN — FAST 100K COLLECTOR")
print("="*90)
print("MAIN:",MAIN_CATEGORY)
print("TARGET:",f"{TARGET:,}")
print("WORKERS:",WORKERS)
print("EXISTING:",f"{total:,}")
print("="*90)

while total<TARGET:

    jobs=[]

    for sub,source in SOURCES.items():
        page,finished=get_state(source)
        if not finished and page<=MAX_ARCHIVE_PAGES:
            jobs.append((source,sub,page))

    if not jobs:
        print("NO MORE ARCHIVE PAGES")
        break

    found={}

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:

        futures={
            ex.submit(fetch,archive_url(source,page)):
            (source,sub,page)
            for source,sub,page in jobs
        }

        for future in as_completed(futures):

            source,sub,page=futures[future]

            try:
                status,html,final=future.result()
            except:
                continue

            if status==200 and html:

                article_urls=extract_links(
                    html,
                    archive_url(source,page))

                if article_urls:

                    for u in article_urls:
                        found.setdefault(u,set()).add(sub)

                    set_state(source,page+1,0)

                else:
                    set_state(source,page,1)

            elif status==429:
                set_state(source,page,0)

            elif status in (500,502,503,504,0):
                set_state(source,page,0)

            else:
                set_state(source,page+1,0)

    if not found:
        time.sleep(5)
        continue

    urls=list(found)

    old=existing_urls(urls)
    urls=[u for u in urls if u not in old]

    if not urls:
        continue

    remaining=TARGET-total
    urls=urls[:remaining+200]

    print(
        f"FOUND {len(urls):,} | "
        f"TOTAL {total:,}/{TARGET:,}")

    buffer=[]

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:

        futures={
            ex.submit(
                parse_article,
                url,
                list(found[url])
            ):url
            for url in urls
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="ARTICLES"):

            try:
                data=future.result()
            except:
                data=None

            if data:
                buffer.append(data)

            if len(buffer)>=DB_BATCH:

                save_batch(buffer)
                buffer=[]

                total=count_articles()

                print(
                    f"SAVED {total:,}/{TARGET:,}")

                if total%CHECKPOINT==0 or total>=TARGET:
                    export_csv()

                if total>=TARGET:
                    break

    if buffer:
        save_batch(buffer)

    total=count_articles()

    if total%CHECKPOINT==0 or total>=TARGET:
        export_csv()

    print(
        f"TOTAL {total:,}/{TARGET:,} | "
        f"TIME {(time.time()-start)/3600:.2f}h")

export_csv()

print("="*90)
print("DONE")
print("TOTAL:",f"{count_articles():,}")
print("TIME:",f"{(time.time()-start)/3600:.2f} HOURS")
print("CSV:",CSV_PATH)
print("DB:",DB_PATH)
print("="*90)

Mounted at /content/drive
AL-AIN — FAST 100K COLLECTOR
MAIN: اقتصاد
TARGET: 100,000
WORKERS: 20
EXISTING: 0
FOUND 76 | TOTAL 0/100,000


ARTICLES:   0%|          | 0/76 [00:00<?, ?it/s]

TOTAL 76/100,000 | TIME 0.00h
FOUND 54 | TOTAL 76/100,000


ARTICLES:   0%|          | 0/54 [00:00<?, ?it/s]

TOTAL 130/100,000 | TIME 0.00h
FOUND 54 | TOTAL 130/100,000


ARTICLES:   0%|          | 0/54 [00:00<?, ?it/s]

... (972 outputs left out)

TOTAL 21,794/100,000 | TIME 0.85h
FOUND 27 | TOTAL 21,794/100,000


ARTICLES:   0%|          | 0/27 [00:00<?, ?it/s]

TOTAL 21,821/100,000 | TIME 0.85h
FOUND 27 | TOTAL 21,821/100,000


ARTICLES:   0%|          | 0/27 [00:00<?, ?it/s]